# Opening-Range Breakout Momentum: Research Notebook

**Hypothesis under test:** when a liquid US stock breaks its first 15-minute opening
range with unusually high volume, the breakout direction has a statistically
significant tendency to continue over the following 30-60 minutes.

This notebook runs Phases 1-5 of the research process end to end. It requires cached
1-minute bar data (see `README.md` -> Setup -> run `scripts/fetch_data.py` first, which
in turn requires `ALPACA_API_KEY`/`ALPACA_SECRET_KEY`). Every code cell here calls into
the tested modules under `data/`, `strategies/`, `backtest/`, `analysis/`, and
`research/` -- this notebook orchestrates, it does not reimplement any logic.

**Discipline reminder (read before running):** the goal is to determine whether the
edge exists, not to make the numbers look good. The validation split is set up front in
the next section and the **test period must not be touched until Phase 4**. If you find
yourself re-running Phase 1/2/3 after peeking at test-period results, stop -- that
invalidates the out-of-sample test.

In [ ]:
import sys
sys.path.insert(0, "..")

import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np

from config import (
    UNIVERSE, DEFAULT_START_DATE, DEFAULT_END_DATE,
    DEFAULT_PARAMS, DEFAULT_EXECUTION, DEFAULT_PORTFOLIO, StrategyParams, ExecutionParams,
)
from data.loader import AlpacaBarLoader
from data.preprocessing import clean_symbol_bars, get_trading_days, detect_price_discontinuities
from strategies.opening_range import compute_opening_range, generate_signal
from backtest.engine import run_backtest, run_direction_control
from backtest.portfolio import Portfolio
from analysis.performance import compute_performance_summary, summary_to_dict, trade_return_distribution, daily_equity_curve
from analysis.statistics import bootstrap_ci, block_bootstrap_ci, permutation_test_two_groups, multiple_testing_correction
from analysis.visualization import (
    plot_equity_curve, plot_drawdown, plot_trade_return_distribution,
    plot_metric_by_bucket, plot_bootstrap_distribution,
)
from research.splits import chronological_split, walk_forward_windows

pd.set_option("display.width", 120)
print("Universe:", UNIVERSE)
print("Requested date range:", DEFAULT_START_DATE, "->", DEFAULT_END_DATE)

## 1. Data loading, cleaning, and audit

Loads cached bars for each symbol, restricts to the regular session, reindexes onto the
full per-day minute grid (flagging synthetic fill-in bars), and drops any day with more
than 5% of the session missing. Also runs a discontinuity check to flag any >20%
overnight gap for manual review (legitimate news event vs. a possible bad adjustment).

In [ ]:
trading_days = pd.DatetimeIndex(get_trading_days(DEFAULT_START_DATE, DEFAULT_END_DATE))
loader = AlpacaBarLoader()

bars_by_symbol = {}
for symbol in UNIVERSE:
    raw = loader.get_bars(symbol, DEFAULT_START_DATE, DEFAULT_END_DATE)
    if raw.empty:
        print(f"[{symbol}] NO CACHED DATA -- run scripts/fetch_data.py first.")
        continue
    clean, dropped = clean_symbol_bars(raw, trading_days)
    bars_by_symbol[symbol] = clean
    print(f"[{symbol}] {len(clean):,} clean bars, {len(dropped)} day(s) dropped for low coverage, "
          f"{clean['is_synthetic'].sum() if 'is_synthetic' in clean else 0} synthetic fill-in bars")

    gaps = detect_price_discontinuities(raw)
    if len(gaps):
        print(f"[{symbol}] {len(gaps)} overnight gap(s) > 20% flagged for manual review:")
        print(gaps)

assert bars_by_symbol, "No data loaded for any symbol -- see README Setup section." 

## 2. Chronological train / validation / test split

60% / 20% / 20% by NYSE trading day. **The test period is not loaded into any Phase
1-3 analysis below** -- it is only used in the explicit Phase 4 section.

In [ ]:
split = chronological_split(DEFAULT_START_DATE, DEFAULT_END_DATE, train_frac=0.6, val_frac=0.2)
print("Train:     ", split.train_start, "->", split.train_end)
print("Validation:", split.val_start, "->", split.val_end)
print("Test:      ", split.test_start, "->", split.test_end, "  <-- untouched until Phase 4")

## 3. Phase 1 -- Baseline

The simplest possible version of the hypothesis, `config.DEFAULT_PARAMS` /
`config.DEFAULT_EXECUTION` / `config.DEFAULT_PORTFOLIO` exactly as specified in the
research brief, **no optimization**, evaluated on train+validation only.

> Does opening-range breakout momentum have positive expectancy at all?

In [ ]:
phase1_result = run_backtest(
    bars_by_symbol, DEFAULT_PARAMS, DEFAULT_EXECUTION, DEFAULT_PORTFOLIO,
    start=split.train_start, end=split.val_end, require_volume_baseline=False,
)

print(f"Days considered: {phase1_result.n_days_considered}")
print(f"Days skipped (incomplete OR): {phase1_result.n_days_skipped_no_or}")
print(f"Signals generated: {len(phase1_result.signals)}")
print(f"Trades filled: {len(phase1_result.trades)}")

phase1_summary = compute_performance_summary(
    phase1_result.trades, DEFAULT_PORTFOLIO.starting_equity, split.train_start, split.val_end
)
pd.Series(summary_to_dict(phase1_summary))

In [ ]:
trade_return_distribution(phase1_result.trades)

In [ ]:
plot_equity_curve(phase1_result.equity_curve, title="Phase 1 Baseline -- Equity Curve (train+val)")
plot_drawdown(phase1_result.equity_curve, title="Phase 1 Baseline -- Drawdown (train+val)")
plot_trade_return_distribution(phase1_result.trades)
print("Saved to reports/")

### Phase 1 significance: direction-randomized control

Compares real trades against a control group built from the *same* signals (same
timing/price/OR levels) with direction randomly re-assigned 50/50 and run through the
*identical* execution/cost pipeline. This isolates whether the TRUE breakout direction
adds value beyond the execution model's own structural asymmetries (a 2:1 R:R target
plus a conservative same-bar tie-break do not average to exactly zero even with zero
real edge -- see `README.md` -> Testing philosophy, and
`tests/test_engine_integration.py`, which caught a bug in an earlier version of this
control).

In [ ]:
phase1_control = run_direction_control(
    bars_by_symbol, phase1_result.signals, DEFAULT_PARAMS, DEFAULT_EXECUTION, DEFAULT_PORTFOLIO, seed=7
)

perm = permutation_test_two_groups(
    phase1_result.trades["r_multiple"].values, phase1_control["r_multiple"].values, n_perm=20_000, random_state=1
)
print(f"Real mean R-multiple:    {phase1_result.trades['r_multiple'].mean():.4f}")
print(f"Control mean R-multiple: {phase1_control['r_multiple'].mean():.4f}")
print(f"Observed difference:     {perm.observed_stat:.4f}")
print(f"Permutation p-value:     {perm.p_value:.4f}  (n_perm={perm.n_perm})")

boot_net = block_bootstrap_ci(phase1_result.trades, value_col="net_pnl", block_col="day", n_boot=10_000)
print(f"\nBlock-bootstrap (by day) 95% CI on mean net P&L per trade: "
      f"${boot_net.point_estimate:.2f} [${boot_net.ci_low:.2f}, ${boot_net.ci_high:.2f}]")

### Phase 1 interim read

Fill this in honestly once real numbers are above. Do **not** proceed to Phase 2 by
default -- the research brief is explicit: *"If the baseline shows evidence of an edge,
investigate..."*. If net expectancy is at or below zero, or the permutation p-value is
not significant, the honest next step is to document that finding (Phase 3 can still
run for completeness/reporting, but Phase 2's conditional search should be read as
exploratory-only, not as a path to rescue the hypothesis) and move toward the final
conclusion rather than mining Phase 2 variables for a profitable subset.

## 4. Phase 2 -- Conditional Analysis

*Only meaningful to over-interpret if Phase 1 showed evidence of an edge.* Examines
whether performance depends on variables that could **plausibly** affect a breakout
continuation, each with a stated reason -- this is not a blind grid search:

- **Relative volume** -- the hypothesis is specifically about *unusually high* volume;
  if the effect is real, it should show up more strongly (or only) in high-relative-
  volume trades.
- **Opening-range width** -- a very narrow range may signal indecision/low
  information content; a very wide range may already have exhausted the move.
- **Time of breakout** -- later breakouts (closer to the 11:00 cutoff) have less
  runway before the 60-minute holding cap and may behave differently from breakouts
  in the first few minutes after the range.
- **Distance from VWAP** -- a breakout aligned with the VWAP trend may differ from one
  fighting it.
- **Individual ticker** -- SPY/QQQ (index-like, mean-reverting tendencies) vs.
  single-name momentum stocks (NVDA/META/AMZN) could plausibly behave differently.
- **Day of week** -- Monday/Friday session-structure effects are a commonly studied
  (if often overstated) seasonality.

Every test performed in this section is collected into one family and corrected for
multiple comparisons in Phase 3 -- do not report any single one of these as significant
in isolation.

In [ ]:
trades = phase1_result.trades.copy()

# Relative volume quintile (only defined for trades where a baseline was available).
has_rv = trades["relative_volume"].notna()
trades.loc[has_rv, "rel_volume_bucket"] = pd.qcut(trades.loc[has_rv, "relative_volume"], 4, duplicates="drop")

# Opening-range width quintile, hour-of-day bucket, ticker, day-of-week.
trades["or_width_bucket"] = pd.qcut(trades["or_width"], 4, duplicates="drop")
trades["breakout_hour"] = pd.to_datetime(trades["entry_time"]).dt.hour
trades["day_of_week"] = pd.to_datetime(trades["entry_time"]).dt.day_name()

bucket_cols = ["rel_volume_bucket", "or_width_bucket", "breakout_hour", "symbol", "day_of_week"]
phase2_pvalues = {}

for col in bucket_cols:
    sub = trades.dropna(subset=[col])
    if sub[col].nunique() < 2:
        continue
    plot_metric_by_bucket(sub, col, value_col="r_multiple", filename=f"phase2_{col}.png")
    groups = [g["r_multiple"].values for _, g in sub.groupby(col) if len(g) >= 5]
    if len(groups) >= 2:
        # Compare the two extreme groups by mean r_multiple as the headline test per variable.
        means = sub.groupby(col)["r_multiple"].mean().sort_values()
        lo_key, hi_key = means.index[0], means.index[-1]
        lo = sub[sub[col] == lo_key]["r_multiple"].values
        hi = sub[sub[col] == hi_key]["r_multiple"].values
        result = permutation_test_two_groups(hi, lo, n_perm=10_000, random_state=3)
        phase2_pvalues[col] = result.p_value
        print(f"{col}: lowest bucket={lo_key} (n={len(lo)}, mean={lo.mean():.3f}) vs "
              f"highest bucket={hi_key} (n={len(hi)}, mean={hi.mean():.3f}) -> p={result.p_value:.4f}")

print("\nPlots saved to reports/phase2_*.png")

## 5. Phase 3 -- Statistical Testing

Formal significance read, with multiple-testing correction applied across **every**
test performed above (the Phase 1 direction-control test plus every Phase 2 bucket
comparison) -- not just the ones that happen to look significant.

In [ ]:
all_pvalues = {"phase1_direction_control": perm.p_value, **{f"phase2_{k}": v for k, v in phase2_pvalues.items()}}
names = list(all_pvalues.keys())
pvals = list(all_pvalues.values())

reject, adj_p = multiple_testing_correction(pvals, method="fdr_bh", alpha=0.05)
correction_table = pd.DataFrame({"test": names, "raw_p": pvals, "fdr_adjusted_p": adj_p, "significant_at_0.05": reject})
correction_table.sort_values("fdr_adjusted_p")

## 6. Phase 4 -- Out-of-sample testing

Re-runs the **exact same baseline parameters** (no re-tuning based on anything seen
above) on the validation period alone, then -- exactly once -- on the test period. If
Phase 2/3 motivated a specific, pre-registered parameter change (e.g. "only take
trades above the median relative-volume bucket"), that change is applied here as a
single named variant, not searched for.

**Run the test-period cell below at most once.** If the result is unfavorable, that is
the answer -- do not go back and adjust parameters.

In [ ]:
val_result = run_backtest(
    bars_by_symbol, DEFAULT_PARAMS, DEFAULT_EXECUTION, DEFAULT_PORTFOLIO,
    start=split.val_start, end=split.val_end, require_volume_baseline=False,
)
val_summary = compute_performance_summary(val_result.trades, DEFAULT_PORTFOLIO.starting_equity, split.val_start, split.val_end)
print("Validation-only performance (sanity check vs. train+val blend above):")
pd.Series(summary_to_dict(val_summary))

In [ ]:
# TEST PERIOD -- run once. Uncomment when Phases 1-3 are complete and no further
# parameter changes will be made.

# test_result = run_backtest(
#     bars_by_symbol, DEFAULT_PARAMS, DEFAULT_EXECUTION, DEFAULT_PORTFOLIO,
#     start=split.test_start, end=split.test_end, require_volume_baseline=False,
# )
# test_summary = compute_performance_summary(
#     test_result.trades, DEFAULT_PORTFOLIO.starting_equity, split.test_start, split.test_end
# )
# pd.Series(summary_to_dict(test_summary))

## 7. Phase 5 -- Walk-forward testing and robustness

### 7a. Walk-forward windows

Rolling-origin: train on N trading days, test on the following M, roll forward, repeat.
Each test window is used exactly once and never leaks into an earlier window's
training period.

In [ ]:
windows = walk_forward_windows(DEFAULT_START_DATE, split.test_end, train_days=126, test_days=42)
print(f"{len(windows)} walk-forward window(s)")

wf_trades = []
for w in windows:
    r = run_backtest(bars_by_symbol, DEFAULT_PARAMS, DEFAULT_EXECUTION, DEFAULT_PORTFOLIO,
                      start=w.test_start, end=w.test_end, require_volume_baseline=False)
    if not r.trades.empty:
        wf_trades.append(r.trades)

if wf_trades:
    wf_all = pd.concat(wf_trades)
    wf_summary = compute_performance_summary(wf_all, DEFAULT_PORTFOLIO.starting_equity, windows[0].test_start, windows[-1].test_end)
    print("Aggregated out-of-sample walk-forward performance:")
    display(pd.Series(summary_to_dict(wf_summary)))
else:
    print("No trades generated across walk-forward test windows.")

### 7b. Robustness sweeps

A strategy that only works at one suspiciously specific parameter value should be
treated as suspicious, not as a discovery. Sweeps opening-range duration, volume
threshold, stop distance (via reward:risk), max holding period, and cost assumptions,
evaluated on train+validation (never test).

In [ ]:
sweep_results = []

for or_minutes in [10, 15, 20, 30]:
    p = StrategyParams(or_minutes=or_minutes)
    r = run_backtest(bars_by_symbol, p, DEFAULT_EXECUTION, DEFAULT_PORTFOLIO,
                      start=split.train_start, end=split.val_end)
    if not r.trades.empty:
        s = compute_performance_summary(r.trades, DEFAULT_PORTFOLIO.starting_equity, split.train_start, split.val_end)
        sweep_results.append({"param": "or_minutes", "value": or_minutes, "n_trades": s.n_trades,
                               "expectancy_r": s.expectancy_r, "sharpe": s.sharpe, "net_pnl": s.net_pnl})

for rr in [1.0, 1.5, 2.0, 3.0]:
    p = StrategyParams(reward_risk=rr)
    r = run_backtest(bars_by_symbol, p, DEFAULT_EXECUTION, DEFAULT_PORTFOLIO,
                      start=split.train_start, end=split.val_end)
    if not r.trades.empty:
        s = compute_performance_summary(r.trades, DEFAULT_PORTFOLIO.starting_equity, split.train_start, split.val_end)
        sweep_results.append({"param": "reward_risk", "value": rr, "n_trades": s.n_trades,
                               "expectancy_r": s.expectancy_r, "sharpe": s.sharpe, "net_pnl": s.net_pnl})

for max_hold in [30, 45, 60, 90, 120]:
    p = StrategyParams(max_holding_minutes=max_hold)
    r = run_backtest(bars_by_symbol, p, DEFAULT_EXECUTION, DEFAULT_PORTFOLIO,
                      start=split.train_start, end=split.val_end)
    if not r.trades.empty:
        s = compute_performance_summary(r.trades, DEFAULT_PORTFOLIO.starting_equity, split.train_start, split.val_end)
        sweep_results.append({"param": "max_holding_minutes", "value": max_hold, "n_trades": s.n_trades,
                               "expectancy_r": s.expectancy_r, "sharpe": s.sharpe, "net_pnl": s.net_pnl})

for extra_cost_bps in [0, 2, 5, 10]:
    execp = ExecutionParams(half_spread_bps=DEFAULT_EXECUTION.half_spread_bps,
                             slippage_bps=DEFAULT_EXECUTION.slippage_bps + extra_cost_bps,
                             commission_per_share=DEFAULT_EXECUTION.commission_per_share,
                             reg_fee_bps_on_sell=DEFAULT_EXECUTION.reg_fee_bps_on_sell)
    r = run_backtest(bars_by_symbol, DEFAULT_PARAMS, execp, DEFAULT_PORTFOLIO,
                      start=split.train_start, end=split.val_end)
    if not r.trades.empty:
        s = compute_performance_summary(r.trades, DEFAULT_PORTFOLIO.starting_equity, split.train_start, split.val_end)
        sweep_results.append({"param": "extra_slippage_bps", "value": extra_cost_bps, "n_trades": s.n_trades,
                               "expectancy_r": s.expectancy_r, "sharpe": s.sharpe, "net_pnl": s.net_pnl})

pd.DataFrame(sweep_results)

## 8. Failure-mode checklist (work through explicitly if Phase 1-3 show an edge)

- [ ] **Look-ahead bias**: re-read `data/preprocessing.py` VWAP/baseline causality tests
      and `strategies/opening_range.py`'s `test_signal_scan_never_uses_future_bars` --
      still passing?
- [ ] **Overfitting**: does the edge survive Phase 5's parameter sweeps, or does it
      collapse outside a narrow band?
- [ ] **Data leakage**: was the test period genuinely never used to select parameters
      or read before Phase 4/the walk-forward run?
- [ ] **Survivorship bias**: acknowledged limitation (fixed, currently-liquid
      universe) -- does the conclusion implicitly assume it generalizes beyond these
      five names? It shouldn't.
- [ ] **Unrealistic execution**: re-check the cost sweep in 7b -- does the edge survive
      2-3x the assumed spread/slippage?
- [ ] **Selection bias**: was the opening-range/breakout definition itself chosen
      after looking at results, or fixed in advance per the research brief? (It was
      fixed in advance here.)
- [ ] **Multiple-testing problems**: is the reported significance the FDR-adjusted
      p-value from Phase 3, not a cherry-picked raw p-value from Phase 2?
- [ ] **Regime dependence**: does the walk-forward equity curve show the edge
      concentrated in one window/regime rather than being reasonably persistent?

## 9. Final conclusion

*(Fill in once Phases 1-5 have actually run against real data. State the answer
plainly -- if the answer is "no", say no.)*

**Question:** Is there convincing evidence that high-volume opening-range breakouts
provide a tradable intraday edge after realistic transaction costs?

**Answer:** _TBD -- pending real data (see README.md -> Status)._